# Creating Object

In [ ]:
from main import *

In [ ]:
Ar = 1.8e13 / np.sqrt(1000.0)
Ea = 17614.0 * 8.314e3
test = chamber_solver(5, 3e6, 0.3, 1, Ar, Ea)

J0, eigenvalues = test.stiffness()

x, states = test.solve(dx=1e-9, Y_tol=1e-6)
E, Rs, dx = test.stability_along_solution(x, states)

state0, R, gamma, h1, h2, h3 = test.initial_conditions()
dY = np.max(np.abs(np.diff(states[:, 3:], axis=0)), axis=1)
jump = np.argmax(dY)

case_names = ["inlet", "jump", "outlet"]
case_states = [states[0], states[jump], states[-1]]

print(f"{'case':<12}{'rho':>9}{'T':>9}{'V':>9}{'Y1':>7}{'Y2':>7}{'Y3':>7}"
      f"{'|l1|':>12}{'|l2|':>12}{'|l3|':>12}{'|l4|':>12}{'|l5|':>12}{'|l6|':>12}"
      f"{'R_s':>12}{'dx_max(m)':>13}")
print("-" * 180)

for name, u in zip(case_names, case_states):
    J = test.jacobian(u, R, gamma, h1, h2, h3)
    ev = np.sort(np.abs(la.eigvals(J)))[::-1]
    nz = ev[ev > 1e-8 * ev[0]]
    Rs_i = nz[0] / nz[-1] if nz.size > 1 else np.nan
    dx_i = 2.0 / nz[0]
    Ep = np.pad(ev, (0, max(0, 6 - len(ev))), constant_values=np.nan)
    print(f"{name:<12}{u[0]:9.3f}{u[1]:9.1f}{u[2]:9.2f}{u[3]:7.3f}{u[4]:7.3f}{u[5]:7.3f}"
          + "".join(f"{e:12.3e}" for e in Ep[:6])
          + f"{Rs_i:12.3e}{dx_i:13.3e}")

print(f"\njump index = {jump}, x = {x[jump]*1e6:.4g} um, T = {states[jump,1]:.4g} K")